# 01c — Generate AI Reviews with Open Models (Hosted API)

**Run once per generator.** Calls the three non-OpenAI generators through the **HuggingFace Inference Providers router** — an OpenAI-compatible endpoint that auto-routes to a hosted provider (Together / Fireworks / DeepInfra / Novita / Groq / …). No GPU needed; parallel API calls finish a 2,500-review run in minutes instead of hours.

| slot | model (HF repo id) |
|---|---|
| `deepseek` | `deepseek-ai/DeepSeek-V4-Flash` |
| `gemma` | `google/gemma-4-31B-it` |
| `qwen` | `Qwen/Qwen3.6-35B-A3B` |

> The original proposal listed IBM Granite as the 4th generator, but no Granite model is served through the router, so it's swapped for DeepSeek-V4-Flash. Update the proposal's generator table to match.

**To use:** set `GENERATOR` in the config cell, then Run All. Re-run with a different `GENERATOR` for each model. Each run appends rows tagged by the `generator` column to the shared `ai_reviews.csv` and is resume-safe.

**Auth & providers:** needs an `HF_TOKEN` (fine-grained, with *"Make calls to Inference Providers"* permission). You must also **enable providers** at https://hf.co/settings/inference-providers — otherwise calls fail with *"not supported by any provider you have enabled."* The preflight cell makes one synchronous call and prints the full error if anything is misconfigured, before the big run starts.

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/ECS111FinalProject'
HUMAN_PATH = os.path.join(PROJECT_DIR, 'data', 'raw', 'human_reviews.csv')
OUT_PATH = os.path.join(PROJECT_DIR, 'data', 'generated', 'ai_reviews.csv')
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
assert os.path.exists(HUMAN_PATH), f'missing {HUMAN_PATH} — run 01_collect_human_reviews first'
print('Input :', HUMAN_PATH)
print('Output:', OUT_PATH)

In [ ]:
!pip install -q -U openai

In [ ]:
from getpass import getpass
os.environ['HF_TOKEN'] = getpass('HF token (with inference permissions): ')

## Config — pick the generator

Set `GENERATOR` to one of `granite` / `gemma` / `qwen`. `BASE_URL` points at the HF router; you can swap it for any OpenAI-compatible provider (and adjust the model id) if a model isn't routable.

In [ ]:
GENERATOR = 'deepseek'   # <-- change to 'gemma' or 'qwen' and re-run for the others
BASE_URL = 'https://router.huggingface.co/v1'

# generator slot -> model id as the router expects it (HF repo id).
# NOTE: if a call fails with "not supported by any provider you have enabled",
# enable providers at https://hf.co/settings/inference-providers (or pin one
# by appending a suffix, e.g. 'deepseek-ai/DeepSeek-V4-Flash:novita').
GENERATOR_MODELS = {
    'deepseek': 'deepseek-ai/DeepSeek-V4-Flash',
    'gemma': 'google/gemma-4-31B-it',
    'qwen': 'Qwen/Qwen3.6-35B-A3B',
}
MODEL = GENERATOR_MODELS[GENERATOR]

N = 2500
SEED = 42
WORKERS = 8
TEMPERATURE = 1.0
TOP_P = 0.95

COLS = ['review_id', 'category', 'parent_asin', 'rating', 'title',
        'text', 'timestamp', 'label', 'generator']
print(f'generator={GENERATOR}  model={MODEL}  base_url={BASE_URL}')

## Prompt construction

Same template as the OpenAI notebook. Target word count is sampled from the human length distribution so AI reviews aren't separable by length alone.

In [ ]:
import random

PROMPT_TEMPLATE = (
    'Write a {rating}-star Amazon product review for: {product_title}\n'
    'Category: {category}\n'
    'Length: about {target_word_count} words.\n'
    'Voice: a real customer who bought this product. No emojis, no hashtags.\n'
    'Output only the review body — no title, no preamble.'
)


def sample_word_count(rng, human_lengths):
    return rng.choice(human_lengths)


def build_prompt(product_title, category, rating, target_word_count):
    return PROMPT_TEMPLATE.format(
        product_title=product_title or 'this product',
        category=str(category).replace('_', ' '),
        rating=int(round(rating)) if rating else 5,
        target_word_count=target_word_count,
    )

## Prepare the product list (resume-safe)

In [ ]:
import pandas as pd
from pathlib import Path

rng = random.Random(SEED)
human_df = pd.read_csv(HUMAN_PATH)
human_lengths = human_df['text'].str.split().str.len().tolist()
seeds_df = (
    human_df.drop_duplicates(subset='parent_asin')
            .sample(n=N, random_state=SEED)
            .reset_index(drop=True)
)

done = set()
if Path(OUT_PATH).exists():
    prev = pd.read_csv(OUT_PATH)
    done = set(prev.loc[prev['generator'] == GENERATOR, 'parent_asin'].astype(str))
    if done:
        print(f'resume: {len(done)} already done for {GENERATOR}, skipping those')
seeds_df = seeds_df[~seeds_df['parent_asin'].astype(str).isin(done)].reset_index(drop=True)
print(f'to generate: {len(seeds_df)}')

## Generate — parallel API calls, resume-safe, incremental writes

Threads call the OpenAI-compatible router endpoint. Each finished review is written and flushed immediately, so a disconnect loses at most the in-flight items — just re-run to resume. If the very first calls all error, check the model id is routable and the token has inference permissions.

In [ ]:
import csv
import threading
import traceback
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

from openai import OpenAI
from tqdm.auto import tqdm

client = OpenAI(base_url=BASE_URL, api_key=os.environ['HF_TOKEN'])


def generate_one(prod, target_wc):
    prompt = build_prompt(prod['title'], prod['category'], prod['rating'], target_wc)
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=int(target_wc * 2.5) + 64,
            temperature=TEMPERATURE,
            top_p=TOP_P,
        )
        text = (resp.choices[0].message.content or '').strip()
        if not text:
            return {'_error': 'empty completion', '_etype': 'EmptyCompletion',
                    'parent_asin': prod['parent_asin']}
    except Exception as e:
        return {'_error': f'{type(e).__name__}: {e}', '_etype': type(e).__name__,
                '_traceback': traceback.format_exc(), 'parent_asin': prod['parent_asin']}

    return {
        'review_id': f"{GENERATOR}_{prod['parent_asin']}_{SEED}",
        'category': prod['category'],
        'parent_asin': prod['parent_asin'],
        'rating': prod['rating'],
        'title': prod['title'],
        'text': text,
        'timestamp': '',
        'label': 1,
        'generator': GENERATOR,
    }


# --- preflight: one synchronous call so a misconfig fails loudly before the big run ---
print(f'preflight: model={MODEL!r}  base_url={BASE_URL!r}')
_probe = generate_one(seeds_df.iloc[0].to_dict() if not seeds_df.empty else
                      {'title': 'x', 'category': 'x', 'rating': 5, 'parent_asin': 'probe'}, 40)
if '_error' in _probe:
    print('PREFLIGHT FAILED — fix this before running the full job:')
    print(_probe.get('_traceback', _probe['_error']))
else:
    print('preflight OK — sample output:\n', _probe['text'][:200])


if seeds_df.empty:
    print('nothing to do — all products already generated for', GENERATOR)
elif '_error' in _probe:
    print('\nAborting full run because preflight failed.')
else:
    file_exists = Path(OUT_PATH).exists()
    f = open(OUT_PATH, 'a', newline='', encoding='utf-8')
    writer = csv.DictWriter(f, fieldnames=COLS, quoting=csv.QUOTE_MINIMAL)
    if not file_exists:
        writer.writeheader()
        f.flush()
    write_lock = threading.Lock()

    n_done, n_err = 0, 0
    err_types = Counter()
    printed_errs = 0  # show full detail for the first few errors only
    pbar = tqdm(total=len(seeds_df), desc=f'{GENERATOR} (workers={WORKERS})')

    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futures = [
            ex.submit(generate_one, prod.to_dict(), sample_word_count(rng, human_lengths))
            for _, prod in seeds_df.iterrows()
        ]
        for fut in as_completed(futures):
            row = fut.result()
            pbar.update(1)
            if row is None:
                continue
            if '_error' in row:
                n_err += 1
                err_types[row.get('_etype', 'Unknown')] += 1
                if printed_errs < 5:
                    printed_errs += 1
                    pbar.write(f'[error {n_err}] asin={row["parent_asin"]}: {row["_error"]}')
                    if row.get('_traceback'):
                        pbar.write(row['_traceback'])
                pbar.set_postfix(errors=n_err, types=dict(err_types))
                continue
            with write_lock:
                writer.writerow(row)
                f.flush()
            n_done += 1

    pbar.close()
    f.close()
    print(f'\ndone: wrote {n_done} new reviews to {OUT_PATH} (errors: {n_err})')
    if err_types:
        print('error breakdown:', dict(err_types))


## Quick sanity check

In [ ]:
df = pd.read_csv(OUT_PATH)
sub = df[df['generator'] == GENERATOR]
print(f'{GENERATOR}: {len(sub)} reviews')
print('all generators in file:', df['generator'].value_counts().to_dict())
wc = sub['text'].str.split().str.len()
print(f'word count: min={wc.min()}, max={wc.max()}, mean={wc.mean():.1f}')
print('\nsample:\n')
print(sub['text'].iloc[0])